In [9]:
# code inspired from
# https://www.dlology.com/blog/how-to-do-unsupervised-clustering-with-keras/
# which was inspired from
# Xie, Junyuan, Ross Girshick, and Ali Farhadi. "Unsupervised deep embedding for clustering analysis." International conference on machine learning. PMLR, 2016.
# https://arxiv.org/abs/1511.06335

from keras.datasets import mnist
import numpy as np
from time import time
import numpy as np
import keras.backend as K
from keras.engine.topology import Layer, InputSpec
from keras.layers import Dense, Input
from keras.models import Model
from keras.optimizers import SGD
from keras import callbacks
from keras.initializers import VarianceScaling
from sklearn.cluster import KMeans
import metrics
import pandas as pd
from robotikos.theta import theta_g1, assign
import seaborn as sns
import pylab as pl
from sklearn.manifold import TSNE
np.random.seed(10)

In [10]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

x = np.concatenate((x_train, x_test))
y = np.concatenate((y_train, y_test))  ## labels (used for verification)
x = x.reshape((x.shape[0], -1))
x = np.divide(x, 255.)
n_clusters = len(np.unique(y))
print(n_clusters)
x.shape

10


(70000, 784)

In [11]:
def _autoencoder_(dims, act='relu', init='glorot_uniform'):

  """
  Fully connected auto-encoder model, symmetric.
  Arguments:
      dims: list of number of units in each layer of encoder. dims[0] is input dim, dims[-1] is units in hidden layer.
          The decoder is symmetric with encoder. So number of layers of the auto-encoder is 2*len(dims)-1
      act: activation, not applied to Input, Hidden and Output layers
  return:
      (ae_model, encoder_model), Model of autoencoder and model of encoder
  """
  n_stacks = len(dims) - 1
  # input
  input_img = Input(shape=(dims[0],), name='input')
  x = input_img
  # internal layers in encoder
  for i in range(n_stacks-1):
      x = Dense(dims[i + 1], activation=act, kernel_initializer=init, name='encoder_%d' % i)(x)

  # hidden layer
  encoded = Dense(dims[-1], kernel_initializer=init, name='encoder_%d' % (n_stacks - 1))(x)  # hidden layer, features are extracted from here

  x = encoded
  # internal layers in decoder
  for i in range(n_stacks-1, 0, -1):
      x = Dense(dims[i], activation=act, kernel_initializer=init, name='decoder_%d' % i)(x)

  # output
  x = Dense(dims[0], kernel_initializer=init, name='decoder_0')(x)
  decoded = x
  return Model(inputs=input_img, outputs=decoded, name='AE'), Model(inputs=input_img, outputs=encoded, name='encoder')

In [12]:
## hyperparameters

dims = [x.shape[-1], 500, 500, 2000, 10]
init = VarianceScaling(scale=1. / 3., mode='fan_in',
                           distribution='uniform')
pretrain_optimizer = SGD(lr=1, momentum=0.9)
pretrain_epochs = 300
batch_size = 256


In [13]:
class ClusteringLayer(Layer):
    """
    Clustering layer converts input sample (feature) to soft label, i.e. a vector that represents the probability of the
    sample belonging to each cluster. The probability is calculated with student's t-distribution.

    # Example
    ```
        model.add(ClusteringLayer(n_clusters=10))
    ```
    # Arguments
        n_clusters: number of clusters.
        weights: list of Numpy array with shape `(n_clusters, n_features)` witch represents the initial cluster centers.
        alpha: degrees of freedom parameter in Student's t-distribution. Default to 1.0.
    # Input shape
        2D tensor with shape: `(n_samples, n_features)`.
    # Output shape
        2D tensor with shape: `(n_samples, n_clusters)`.
    """

    def __init__(self, n_clusters, weights=None, alpha=1.0, **kwargs):
        if 'input_shape' not in kwargs and 'input_dim' in kwargs:
            kwargs['input_shape'] = (kwargs.pop('input_dim'),)
        super(ClusteringLayer, self).__init__(**kwargs)
        self.n_clusters = n_clusters
        self.alpha = alpha
        self.initial_weights = weights
        self.input_spec = InputSpec(ndim=2)

    def build(self, input_shape):
        assert len(input_shape) == 2
        input_dim = input_shape[1]
        self.input_spec = InputSpec(dtype=K.floatx(), shape=(None, input_dim))
        self.clusters = self.add_weight(shape=(self.n_clusters, input_dim), initializer='glorot_uniform', name='clusters')
        if self.initial_weights is not None:
            self.set_weights(self.initial_weights)
            del self.initial_weights
        self.built = True

    def call(self, inputs, **kwargs):
        """ student t-distribution, as same as used in t-SNE algorithm.
         Measure the similarity between embedded point z_i and centroid µ_j.
                 q_ij = 1/(1+dist(x_i, µ_j)^2), then normalize it.
                 q_ij can be interpreted as the probability of assigning sample i to cluster j.
                 (i.e., a soft assignment)
        Arguments:
            inputs: the variable containing data, shape=(n_samples, n_features)
        Return:
            q: student's t-distribution, or soft labels for each sample. shape=(n_samples, n_clusters)
        """
        q = 1.0 / (1.0 + (K.sum(K.square(K.expand_dims(inputs, axis=1) - self.clusters), axis=2) / self.alpha))
        q **= (self.alpha + 1.0) / 2.0
        q = K.transpose(K.transpose(q) / K.sum(q, axis=1)) # Make sure each sample's 10 values add up to 1.
        return q

    def compute_output_shape(self, input_shape):
        assert input_shape and len(input_shape) == 2
        return input_shape[0], self.n_clusters

    def get_config(self):
        config = {'n_clusters': self.n_clusters}
        base_config = super(ClusteringLayer, self).get_config()
        return dict(list(base_config.items()) + list(config.items()))


In [14]:
def labels_to_centroids(data, labels):

    unique_labels, unique_counts = np.unique(labels,
                                             return_counts=True)
    
    centroids = np.zeros((len(unique_labels), data.shape[1]))

    for (i, (u, c)) in enumerate(zip(unique_labels, unique_counts)):
        centroids[i] = np.sum(data[labels == u], axis=0) / c

    return centroids


def theta_g2(X, theta, metric='euclidean', num_shuffles=10):
    sum_of_time = 0
    all_centroids = {}
    X_work = X.copy()

    num_centroids = []
    timings = []
    for s in range(num_shuffles):

        np.random.shuffle(X_work)
        X_work_cp = np.ascontiguousarray(X_work)
        t1 = time()
        centroids, _ = theta_g1(X_work_cp, theta=theta, metric=metric)
        t2 = time()        
        timings.append(t2-t1)

        all_centroids[s] = centroids
        num_centroids.append(len(centroids))

    #print('Time for tg1 calls', np.sum(timings))
    sum_of_time += np.sum(timings)
    total_num_centroids = np.sum(num_centroids)
    X_centroids = np.zeros((total_num_centroids, X_work.shape[1]), dtype='f8')

    offset = 0

    for s in range(num_shuffles):

        X_centroids[offset:offset + all_centroids[s].shape[0]] = all_centroids[s]
        offset += all_centroids[s].shape[0]
    
    t1 = time()
    new_centroids, _ = theta_g1(X_centroids, theta=theta, metric=metric)
    t2 = time()
    sum_of_time += t2 - t1
    #print('Time for tg1 on centroids', t2 - t1)
    

    t1 = time()
    final_indices = assign(X, new_centroids, metric=metric)
    t2 = time()
    sum_of_time += t2 - t1
    #print('Time for assign calls', t2 - t1)
    final_centroids = labels_to_centroids(X, final_indices)

    return final_centroids, final_indices, sum_of_time

In [15]:
# computing an auxiliary target distribution
def target_distribution(q):
    weight = q ** 2 / q.sum(0)
    return (weight.T / weight.sum(1)).T

In [16]:
autoencoder, encoder = _autoencoder_(dims, init=init)
autoencoder.load_weights('ae_weights.h5')
z = encoder.predict(x)
z = np.ascontiguousarray(z.astype(np.double))

max_acc = 0
max_centroids = None
test_data = {'acc': [], 'clu_n': [], 'theta': []}
for theta in np.arange(4.0, 5.001, 0.001):
    tcentroids, tlabels, _ = theta_g2(z, theta, num_shuffles=50)
    num_idx_counts = np.zeros((len(tcentroids)))
    test_data['theta'].append(theta)
    test_data['clu_n'].append(len(tcentroids))
    for i in tlabels:
        num_idx_counts[i] += 1
    for i in range(len(tcentroids)-10):
        min_idx = np.argmin(num_idx_counts)
        num_idx_counts[min_idx] = 70000
    #print(tcentroids.shape)
    new_centroids = np.zeros((10, tcentroids.shape[1]))
    idx = 0
    for i in range(len(tcentroids)):
        if num_idx_counts[i] != 70000:
            new_centroids[idx] = tcentroids[i]
            idx += 1

    y_pred = assign(z, new_centroids)
    acc = np.round(metrics.acc(y, y_pred), 5)
    test_data['acc'].append(acc)
    if max_acc < acc:
        max_acc = acc
        max_centroids = new_centroids
    
test_data = pd.DataFrame.from_dict(test_data)
line_plot = sns.lineplot(data=test_data, x='theta', y='acc')
fig = line_plot.get_figure()
fig.savefig('acc_test.png', dpi=300)
pl.close(fig)

line_plot = sns.lineplot(data=test_data, x='theta', y='clu_n')
fig = line_plot.get_figure()
fig.savefig('len_test.png', dpi=300)
pl.close(fig)


    
    

In [17]:
data = {'acc': [], 'nmi': [], 'ari': [], 'method': [], 'epochs': []}

#ThetA step
autoencoder, encoder = _autoencoder_(dims, init=init)
autoencoder.load_weights('ae_weights.h5')
clustering_layer = ClusteringLayer(n_clusters, name='clustering')(encoder.output)
model = Model(inputs=encoder.input,
            outputs=[clustering_layer, autoencoder.output])
model.compile(loss=['kld', 'mse'], loss_weights=[0.1, 1], optimizer=pretrain_optimizer)
autoencoder.load_weights('ae_weights.h5')
model.get_layer(name='clustering').set_weights([max_centroids])
y_pred_last = np.copy(y_pred)

loss = 0
index = 0

maxiter = 4000
update_interval = 100
index_array = np.arange(x.shape[0])

tol = 0.001 # tolerance threshold to stop training
for ite in range(int(maxiter)):
    if ite % update_interval == 0:
        q, _  = model.predict(x, verbose=0)
        p = target_distribution(q)  # update the auxiliary target distribution p

        # evaluate the clustering performance
        y_pred = q.argmax(1)
        acc = np.round(metrics.acc(y, y_pred), 5)
        nmi = np.round(metrics.nmi(y, y_pred, average_method='arithmetic'), 5)
        ari = np.round(metrics.ari(y, y_pred), 5)
        loss = np.round(loss, 5)

        # check stop criterion
        data['acc'].append(acc)
        data['nmi'].append(nmi)
        data['ari'].append(ari)
        data['method'].append('ThetA')
        data['epochs'].append(ite)
        
    idx = index_array[index * batch_size: min((index+1) * batch_size, x.shape[0])]
    loss = model.train_on_batch(x=x[idx], y=[p[idx], x[idx]])
    index = index + 1 if (index + 1) * batch_size <= x.shape[0] else 0

q, _  = model.predict(x, verbose=0)
t_pred = q.argmax(1)
    
autoencoder, encoder = _autoencoder_(dims, init=init)
autoencoder.load_weights('ae_weights.h5')
clustering_layer = ClusteringLayer(n_clusters, name='clustering')(encoder.output)
model = Model(inputs=encoder.input,
            outputs=[clustering_layer, autoencoder.output])
model.compile(loss=['kld', 'mse'], loss_weights=[0.1, 1], optimizer=pretrain_optimizer)
kmeans = KMeans(n_clusters=n_clusters, n_init=50)
y_pred = kmeans.fit_predict(z)
model.get_layer(name='clustering').set_weights([kmeans.cluster_centers_])
y_pred_last = np.copy(y_pred)

loss = 0
index = 0

maxiter = 4000
update_interval = 100
index_array = np.arange(x.shape[0])

tol = 0.001 # tolerance threshold to stop training
for ite in range(int(maxiter)):
    if ite % update_interval == 0:
        q, _  = model.predict(x, verbose=0)
        p = target_distribution(q)  # update the auxiliary target distribution p

        # evaluate the clustering performance
        y_pred = q.argmax(1)
        acc = np.round(metrics.acc(y, y_pred), 5)
        nmi = np.round(metrics.nmi(y, y_pred, average_method='arithmetic'), 5)
        ari = np.round(metrics.ari(y, y_pred), 5)
        loss = np.round(loss, 5)

        # check stop criterion
        data['acc'].append(acc)
        data['nmi'].append(nmi)
        data['ari'].append(ari)
        data['method'].append('kmeans++')
        data['epochs'].append(ite)

    idx = index_array[index * batch_size: min((index+1) * batch_size, x.shape[0])]
    loss = model.train_on_batch(x=x[idx], y=[p[idx], x[idx]])
    index = index + 1 if (index + 1) * batch_size <= x.shape[0] else 0

q, _  = model.predict(x, verbose=0)
kp_pred = q.argmax(1)
    
autoencoder, encoder = _autoencoder_(dims, init=init)
autoencoder.load_weights('ae_weights.h5')
clustering_layer = ClusteringLayer(n_clusters, name='clustering')(encoder.output)
model = Model(inputs=encoder.input,
            outputs=[clustering_layer, autoencoder.output])
model.compile(loss=['kld', 'mse'], loss_weights=[0.1, 1], optimizer=pretrain_optimizer)
kmeans = KMeans(n_clusters=n_clusters, n_init=50, init='random')
y_pred = kmeans.fit_predict(z)
model.get_layer(name='clustering').set_weights([kmeans.cluster_centers_])
y_pred_last = np.copy(y_pred)

loss = 0
index = 0

maxiter = 4000
update_interval = 140
index_array = np.arange(x.shape[0])

tol = 0.001 # tolerance threshold to stop training
for ite in range(int(maxiter)):
    if ite % update_interval == 0:
        q, _  = model.predict(x, verbose=0)
        p = target_distribution(q)  # update the auxiliary target distribution p

        # evaluate the clustering performance
        y_pred = q.argmax(1)
        acc = np.round(metrics.acc(y, y_pred), 5)
        nmi = np.round(metrics.nmi(y, y_pred, average_method='arithmetic'), 5)
        ari = np.round(metrics.ari(y, y_pred), 5)
        loss = np.round(loss, 5)

        # check stop criterion
        data['acc'].append(acc)
        data['nmi'].append(nmi)
        data['ari'].append(ari)
        data['method'].append('kmeans')
        data['epochs'].append(ite)
        
    idx = index_array[index * batch_size: min((index+1) * batch_size, x.shape[0])]
    loss = model.train_on_batch(x=x[idx], y=[p[idx], x[idx]])
    index = index + 1 if (index + 1) * batch_size <= x.shape[0] else 0
    
q, _  = model.predict(x, verbose=0)
k_pred = q.argmax(1)
data = pd.DataFrame.from_dict(data)
    
    

In [18]:
line_plot = sns.lineplot(data=data, x='epochs', y='acc', hue='method')
fig = line_plot.get_figure()
fig.savefig('acc_deep.png', dpi=300)
pl.close(fig)

line_plot = sns.lineplot(data=data, x='epochs', y='nmi', hue='method')
fig = line_plot.get_figure()
fig.savefig('nmi_deep.png', dpi=300)
pl.close(fig)

line_plot = sns.lineplot(data=data, x='epochs', y='ari', hue='method')
fig = line_plot.get_figure()
fig.savefig('ari_deep.png', dpi=300)
pl.close(fig)

In [28]:
x_embedded = TSNE(n_components=2, perplexity=50.0).fit_transform(z)
print(x_embedded.shape)


(70000, 2)


In [29]:
fig = pl.figure()
pl.title('ThetA')
pl.scatter(x_embedded[..., 0], x_embedded[..., 1], c=t_pred, cmap='tab20b', alpha = 0.5)
pl.savefig('ThetA_tsne_tab20b_50.png', dpi=300)
pl.close(fig)

fig = pl.figure()
pl.title('kmeans')
pl.scatter(x_embedded[..., 0], x_embedded[..., 1], c=k_pred, cmap='tab20b', alpha = 0.5)
pl.savefig('kmeans_tsne_tab20b_50.png', dpi=300)
pl.close(fig)

fig = pl.figure()
pl.title('kmeans++')
pl.scatter(x_embedded[..., 0], x_embedded[..., 1], c=kp_pred, cmap='tab20b', alpha = 0.5)
pl.savefig('kmeans++_tsne_tab20b_50.png', dpi=300)
pl.close(fig)